# 파일 불러오기

- [메인] smartphone (분석용)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("/sonamu/final_project/data/2019-Nov/2019-Oct.csv")

In [ ]:
# 지수 -> 소수점(2) 표기
pd.options.display.float_format = '{:.2f}'.format

In [ ]:
plt.rc('font', family='Malgun Gothic')
plt.rcParams['axes.unicode_minus'] = False

In [ ]:
print(df.shape)

In [ ]:
smartphone_df = df[df['category_code'] == 'electronics.smartphone'].copy()

In [ ]:
chunk_iter = pd.read_csv("/sonamu/final_project/data/2019-Nov/2019-Nov.csv", chunksize=1000000)
smartphone_chunks = []

for chunk in chunk_iter:
    filtered = chunk[chunk['category_code'] == 'electronics.smartphone'].copy()
    smartphone_chunks.append(filtered)
    
smartphone_df2 = pd.concat(smartphone_chunks, axis=0)

In [ ]:
print(smartphone_df.shape)
print(smartphone_df2.shape)

In [ ]:
df = smartphone_df
df2 = smartphone_df2

del smartphone_df

In [ ]:
import gc
gc.collect()

In [ ]:
print(df.shape)
print(df2.shape)

In [ ]:
def optimize_memory(df):
    for col in df.columns:
        if df[col].dtype == 'float64':
            df[col] = df[col].astype('float32')
        if df[col].dtype == 'int64':
            df[col] = df[col].astype('int32')
    return df

df = optimize_memory(df)
df2 = optimize_memory(df2)

# 데이터 확인

### 기본

In [ ]:
print(df.head())
print(df.shape)

In [ ]:
print(df.info())

In [ ]:
print(df2.head())
print(df2.shape)

In [ ]:
print(df2.info())

In [ ]:
print(df.columns.tolist())

In [ ]:
print(df.isna().sum())
print("------")
print(df2.isna().sum())

In [ ]:
print(df.isna().mean() * 100)
print("------")
print(df2.isna().mean() * 100)

In [ ]:
print(df.duplicated().sum())
print("------")
print(df2.duplicated().sum())

In [ ]:
print(df.nunique())
print("------")
print(df2.nunique())

In [ ]:
print(df['category_code'].isna().sum())
print(df2['category_code'].isna().sum())

In [ ]:
print(df['brand'].isna().sum())
print(df2['brand'].isna().sum())

In [ ]:
df[~df['category_code'].isna()].head(10)

In [ ]:
df['event_type'].value_counts()

In [ ]:
df['brand'].value_counts()

### 브랜드

In [ ]:
brand_view = (
    df[df['event_type'] == 'view']
    .groupby('brand')
    .size()
    .sort_values(ascending=False)
    .reset_index(name='view_count')
)

brand_view.head(10)

In [ ]:
brand_view2 = (
    df2[df2['event_type'] == 'view']
    .groupby('brand')
    .size()
    .sort_values(ascending=False)
    .reset_index(name='view_count')
)

brand_view2.head(10)

In [ ]:
#비율
brand_view_ratio = brand_view.copy()
brand_view_ratio['ratio'] = brand_view_ratio['view_count'] / brand_view_ratio['view_count'].sum()

brand_view_ratio2 = brand_view2.copy()
brand_view_ratio2['ratio'] = brand_view_ratio2['view_count'] / brand_view_ratio2['view_count'].sum()

print(brand_view_ratio.head(10))
print("------")
print(brand_view_ratio2.head(10))

In [ ]:
top10 = brand_view.head(10)

plt.figure()
plt.bar(top10['brand'], top10['view_count'])
plt.xticks(rotation=45)
plt.title('조회수 기준 스마트폰 상위 10개 브랜드(10월)')
plt.show()

top10_2 = brand_view2.head(10)

plt.figure()
plt.bar(top10_2['brand'], top10_2['view_count'])
plt.xticks(rotation=45)
plt.title('조회수 기준 스마트폰 상위 10개 브랜드(11월)')
plt.show()

# 기본 전처리

In [ ]:
# 중복 제거
df = df.drop_duplicates()
df2 = df2.drop_duplicates()

In [ ]:
# 브랜드 결측치 (unknown)
df['brand'] = df['brand'].fillna('unknown')
df2['brand'] = df2['brand'].fillna('unknown')

In [ ]:
# category_code 결측치 (unknown)
df['category_code'] = df['category_code'].fillna('unknown')
df2['category_code'] = df2['category_code'].fillna('unknown')

### 날짜

In [ ]:
df['event_time'].head()

In [ ]:
df['event_time'] = pd.to_datetime(df['event_time'], utc=True).dt.tz_convert('Asia/Seoul').dt.tz_localize(None)
df2['event_time'] = pd.to_datetime(df2['event_time'], utc=True).dt.tz_convert('Asia/Seoul').dt.tz_localize(None)

# datetime으로 타입 변환, utc → 한국 시간으로 변환

### 통계적 이상치 제거

1. 반복구매 이상치 후보 플래그
    - 동일고객이 동일세션에서 동일한 상품을 반복해서 구매할때
- 카테고리별 중복구매수 iqr 상한값을 구한 후
- 동일유저 + 동일상품 + 동일 세션에서 각 카테고리별 상한값 초과 반복 구매한 세션 중 짧은시간(0~60초) 안에 재 구매한 세션을 최종 이상후보 세션이라고 가정
- 단순 중복과 통계적 이상치 구분(근거는 최종 통합 코드에만 첨부)

In [ ]:
### 10월 (purchase_sess_out_10)

# purchase 행만 추출
df_purchase_10 = df[df['event_type'] == 'purchase'].copy()

# 같은 유저 + 상품 + 세션 기준 시간순 정렬
df_purchase_10 = df_purchase_10.sort_values(['user_id', 'user_session', 'product_id', 'event_time']).copy()

# 바로 이전 purchase 시점 구하기
df_purchase_10['prev_time'] = df_purchase_10.groupby(
    ['user_id', 'user_session', 'product_id']
)['event_time'].shift(1)

# 재구매 간격(초) 계산
df_purchase_10['gap_sec'] = (
    df_purchase_10['event_time'] - df_purchase_10['prev_time']
).dt.total_seconds()

# 60초 이내 재구매가 존재하는 세션/상품 조합만 
short_gap_sessions_10 = df_purchase_10.loc[
    df_purchase_10['gap_sec'].between(0, 60), 
    ['user_id', 'product_id', 'user_session']
].drop_duplicates()


# 동일 유저 + 동일 상품 + 동일 세션 기준 구매 정보 집계 
purchase_sess_10 = (
    df[df['event_type'] == 'purchase']
    .groupby(['user_id', 'product_id', 'user_session'])
    .agg(
        buy_cnt=('event_type', 'size'),
        category_code=('category_code', lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else 'unknown')
    )
    .reset_index()
)

# 2회 이상 반복 구매한 경우만 추출
purchase_sess_2_10 = purchase_sess_10[purchase_sess_10['buy_cnt'] >= 2].copy()

# 위에서 계산한 것 중 60초 이내 재구매 명단(short_gap)에 있는 조합만 남기기
purchase_sess_2_10 = purchase_sess_2_10.merge(
    short_gap_sessions_10, 
    on=['user_id', 'product_id', 'user_session'], 
    how='inner'
)


# 카테고리별 중복 구매수 IQR 상한값 계산
cat_iqr_10 = (
    purchase_sess_2_10
    .groupby('category_code')['buy_cnt']
    .agg(
        q1=lambda x: x.quantile(0.25),
        q3=lambda x: x.quantile(0.75))
    .reset_index())

cat_iqr_10['iqr'] = cat_iqr_10['q3'] - cat_iqr_10['q1']
cat_iqr_10['upper'] = cat_iqr_10['q3'] + 1.5 * cat_iqr_10['iqr']

# 방금 구한 purchase_sess_2에 카테고리별 upper 붙이기
purchase_sess_2_10 = purchase_sess_2_10.merge(
    cat_iqr_10[['category_code', 'upper']],
    on='category_code',
    how='left')

# 카테고리별 IQR 상한값 초과인 조합만 추출
purchase_sess_out_10 = purchase_sess_2_10[
    purchase_sess_2_10['buy_cnt'] > purchase_sess_2_10['upper']
].copy()

print("동일 유저 + 동일 상품 + 동일 세션 기준 2회 이상 반복 구매 조합 수:", len(purchase_sess_2_10))
print("카테고리별 IQR 상한값 초과 조합 수:", len(purchase_sess_out_10))
print("전체 2회 이상 조합 대비 비율:", round(len(purchase_sess_out_10) / len(purchase_sess_2_10) * 100, 2), "%")

display(purchase_sess_out_10.head())

In [ ]:
### 11월 (purchase_sess_out_11)

# purchase 행만 추출
df_purchase_11 = df2[df2['event_type'] == 'purchase'].copy()

# 같은 유저 + 상품 + 세션 기준 시간순 정렬
df_purchase_11 = df_purchase_11.sort_values(['user_id', 'user_session', 'product_id', 'event_time']).copy()

# 바로 이전 purchase 시점 구하기
df_purchase_11['prev_time'] = df_purchase_11.groupby(
    ['user_id', 'user_session', 'product_id']
)['event_time'].shift(1)

# 재구매 간격(초) 계산
df_purchase_11['gap_sec'] = (
    df_purchase_11['event_time'] - df_purchase_11['prev_time']
).dt.total_seconds()

# 60초 이내 재구매가 존재하는 세션/상품 조합만 
short_gap_sessions_11 = df_purchase_11.loc[
    df_purchase_11['gap_sec'].between(0, 60), 
    ['user_id', 'product_id', 'user_session']
].drop_duplicates()


# 동일 유저 + 동일 상품 + 동일 세션 기준 구매 정보 집계 
purchase_sess_11 = (
    df2[df2['event_type'] == 'purchase']
    .groupby(['user_id', 'product_id', 'user_session'])
    .agg(
        buy_cnt=('event_type', 'size'),
        category_code=('category_code', lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else 'unknown')
    )
    .reset_index()
)

# 2회 이상 반복 구매한 경우만 추출
purchase_sess_2_11 = purchase_sess_11[purchase_sess_11['buy_cnt'] >= 2].copy()

# 위에서 계산한 것 중 60초 이내 재구매 명단(short_gap)에 있는 조합만 남기기
purchase_sess_2_11 = purchase_sess_2_11.merge(
    short_gap_sessions_11, 
    on=['user_id', 'product_id', 'user_session'], 
    how='inner'
)


# 카테고리별 중복 구매수 IQR 상한값 계산
cat_iqr_11 = (
    purchase_sess_2_11
    .groupby('category_code')['buy_cnt']
    .agg(
        q1=lambda x: x.quantile(0.25),
        q3=lambda x: x.quantile(0.75))
    .reset_index())

cat_iqr_11['iqr'] = cat_iqr_11['q3'] - cat_iqr_11['q1']
cat_iqr_11['upper'] = cat_iqr_11['q3'] + 1.5 * cat_iqr_11['iqr']

# 방금 구한 purchase_sess_2에 카테고리별 upper 붙이기
purchase_sess_2_11 = purchase_sess_2_11.merge(
    cat_iqr_11[['category_code', 'upper']],
    on='category_code',
    how='left')

# 카테고리별 IQR 상한값 초과인 조합만 추출
purchase_sess_out_11 = purchase_sess_2_11[
    purchase_sess_2_11['buy_cnt'] > purchase_sess_2_11['upper']
].copy()

print("동일 유저 + 동일 상품 + 동일 세션 기준 2회 이상 반복 구매 조합 수:", len(purchase_sess_2_11))
print("카테고리별 IQR 상한값 초과 조합 수:", len(purchase_sess_out_11))
print("전체 2회 이상 조합 대비 비율:", round(len(purchase_sess_out_11) / len(purchase_sess_2_11) * 100, 2), "%")

display(purchase_sess_out_11.head())

2. price (통계적 이상치)
- price가 0 이하인 행들 데이터 누락 판단(삭제 유보) → 플래그 유지 + 목적별 처리

In [ ]:
# 가격 이상치 및 결측치 (IQR 방식 계산)

# 10월
q1_10 = df['price'].quantile(0.25)
q3_10 = df['price'].quantile(0.75)
upper_10 = q3_10 + 3.0 * (q3_10 - q1_10)

df['is_price_error'] = (
    (df['price'] <= 0) | 
    (df['price'].isna()) | 
    (df['price'] > upper_10)
)

# 11월
q1_11 = df2['price'].quantile(0.25)
q3_11 = df2['price'].quantile(0.75)
upper_11 = q3_11 + 3.0 * (q3_11 - q1_11)

df2['is_price_error'] = (
    (df2['price'] <= 0) | 
    (df2['price'].isna()) | 
    (df2['price'] > upper_11)
)

# 파생변수

### 날짜 파생변수
- 시 / 요일 / 요일명
- 시간대

In [ ]:
df['hour'] = df['event_time'].dt.hour #시 추출
df['day_of_week'] = df['event_time'].dt.dayofweek  # 0=월요일, 6=일요일
df['day_name'] = df['event_time'].dt.day_name() #영문 이름 붙이기

In [ ]:
df2['hour'] = df2['event_time'].dt.hour #시 추출
df2['day_of_week'] = df2['event_time'].dt.dayofweek  # 0=월요일, 6=일요일
df2['day_name'] = df2['event_time'].dt.day_name() #영문 이름 붙이기

In [ ]:
# 시간대
def time_segment(hour):
    if 6 <= hour < 12:
        return '오전 (6-12시)'
    elif 12 <= hour < 18:
        return '오후 (12-18시)'
    elif 18 <= hour < 24:
        return '저녁 (18-24시)'
    else:
        return '새벽 (0-6시)'

df['time_segment'] = df['hour'].apply(time_segment)
print(df['time_segment'].value_counts())

In [ ]:
# 시간대
def time_segment(hour):
    if 6 <= hour < 12:
        return '오전 (6-12시)'
    elif 12 <= hour < 18:
        return '오후 (12-18시)'
    elif 18 <= hour < 24:
        return '저녁 (18-24시)'
    else:
        return '새벽 (0-6시)'

df2['time_segment'] = df2['hour'].apply(time_segment)
print(df2['time_segment'].value_counts())

### 반복구매 이상치 후보 플래그

In [ ]:
# 이상치 후보(반복 구매) purchase_sess_out_10 이용

# 해당 리스트 속의 세션/상품 조합만 이상치 플래그 세우기
df = df.merge(
    purchase_sess_out_10[['user_id', 'product_id', 'user_session', 'buy_cnt']],
    on=['user_id', 'product_id', 'user_session'],
    how='left'
)

# 플래그 처리: purchase_count가 1보다 크면 True(이상치), 아니면 False
df['is_outlier'] = df['buy_cnt'].notna() 
df['is_outlier'] = df['is_outlier'].fillna(False)

In [ ]:
# 이상치 후보(반복 구매) purchase_sess_out_11 이용

# 해당 리스트 속의 세션/상품 조합만 이상치 플래그 세우기
df2 = df2.merge(
    purchase_sess_out_11[['user_id', 'product_id', 'user_session', 'buy_cnt']],
    on=['user_id', 'product_id', 'user_session'],
    how='left'
)

# 플래그 처리: purchase_count가 1보다 크면 True(이상치), 아니면 False
df2['is_outlier'] = df2['buy_cnt'].notna() 
df2['is_outlier'] = df2['is_outlier'].fillna(False)

In [ ]:
df_final = df[df['is_outlier'] == False].copy()
df2_final = df2[df2['is_outlier'] == False].copy()

### 상태 식별 플래그(is~) / 활동량 집계 변수(total~)

In [ ]:
### 상태 식별 플래그 (is~)
df_final['is_purchase'] = (df_final['event_type'] == 'purchase').astype(int)
df_final['is_cart'] = (df_final['event_type'] == 'cart').astype(int)

df2_final['is_purchase'] = (df2_final['event_type'] == 'purchase').astype(int)
df2_final['is_cart'] = (df2_final['event_type'] == 'cart').astype(int)

print(df_final['is_purchase'].sum())
print(df_final['is_cart'].sum())
print("------")
print(df2_final['is_purchase'].sum())
print(df2_final['is_cart'].sum())

In [ ]:
# 평균으로 확률 도출 
print(df_final['is_purchase'].mean().round(3))
print(df2_final['is_purchase'].mean().round(3))

In [ ]:
### 활동량 집계 변수 (total~)

total_view = (df_final['event_type'] == 'view').sum()
total_cart = df_final['is_cart'].sum()
total_purchase = df_final['is_purchase'].sum()

# 검증용
len_evt = total_view + total_cart + total_purchase


total_view2 = (df2_final['event_type'] == 'view').sum()
total_cart2 = df2_final['is_cart'].sum()
total_purchase2 = df2_final['is_purchase'].sum()

# 검증용
len_evt2 = total_view2 + total_cart2 + total_purchase2

print(len_evt)
print(df_final['event_type'].shape)
print("------")
print(len_evt2)
print(df2_final['event_type'].shape)

### 결측 패턴 탐색 (category_code & brand)

In [ ]:
# category_code
yes_unknown = df_final[df_final['category_code'] == 'unknown'].copy()
no_unknown = df_final[df_final['category_code'] != 'unknown'].copy()

yes_unknown2 = df2_final[df2_final['category_code'] == 'unknown'].copy()
no_unknown2 = df2_final[df2_final['category_code'] != 'unknown'].copy()

In [ ]:
# 전환율 비교
yes_unknown_cv = yes_unknown['is_purchase'].mean()
no_unknown_cv = no_unknown['is_purchase'].mean()

yes_unknown_cv2 = yes_unknown2['is_purchase'].mean()
no_unknown_cv2 = no_unknown2['is_purchase'].mean()

print(yes_unknown_cv)
print(no_unknown_cv.round(4))
print("------")
print(yes_unknown_cv2)
print(no_unknown_cv2.round(4))

In [ ]:
# brand
yes_brand = df_final[df_final['brand'] != 'unknown'].copy()
no_brand = df_final[df_final['brand'] == 'unknown'].copy()

yes_brand2 = df2_final[df2_final['brand'] != 'unknown'].copy()
no_brand2 = df2_final[df2_final['brand'] == 'unknown'].copy()

print(f"브랜드 있는 그룹 전환율(10월): {yes_brand['is_purchase'].mean():.4f}")
print(f"브랜드 없는 그룹 전환율(10월): {no_brand['is_purchase'].mean():.4f}")
print("------")
print(f"브랜드 있는 그룹 전환율(11월): {yes_brand2['is_purchase'].mean():.4f}")
print(f"브랜드 없는 그룹 전환율(11월): {no_brand2['is_purchase'].mean():.4f}")

### 메인 퍼널(실제 상품 단위) - funnel_df

In [ ]:
# 그룹화
grouped = df_final.groupby(['user_session', 'brand'])
grouped2 = df2_final.groupby(['user_session', 'brand'])

In [ ]:
# 메인 퍼널 로직 (10월)

result = []
for (session, brand), group in grouped:
    group = group.sort_values('event_time')  
    events = group['event_type'].tolist()

    
    # index 찾기
    view_idx = next((i for i, e in enumerate(events) if e == 'view'), None)
    cart_idx = next((i for i, e in enumerate(events) if e == 'cart'), None)
    purchase_idx = next((i for i, e in enumerate(events) if e == 'purchase'), None)
    
    has_view = view_idx is not None
    valid_view_cart = view_idx is not None and cart_idx is not None and view_idx < cart_idx
    valid_cart_purchase = cart_idx is not None and purchase_idx is not None and cart_idx < purchase_idx
    
    result.append([
        session, brand,
        has_view,
        valid_view_cart,
        valid_cart_purchase
    ])

funnel_df = pd.DataFrame(result, columns=[
    'user_session', 'brand',
    'view', 'view_to_cart', 'cart_to_purchase'
])

In [ ]:
# 메인 퍼널 로직 (11월)

result2 = []
for (session, brand), group in grouped2:
    group = group.sort_values('event_time')  
    events = group['event_type'].tolist()

    
    # index 찾기
    view_idx = next((i for i, e in enumerate(events) if e == 'view'), None)
    cart_idx = next((i for i, e in enumerate(events) if e == 'cart'), None)
    purchase_idx = next((i for i, e in enumerate(events) if e == 'purchase'), None)
    
    has_view = view_idx is not None
    valid_view_cart = view_idx is not None and cart_idx is not None and view_idx < cart_idx
    valid_cart_purchase = cart_idx is not None and purchase_idx is not None and cart_idx < purchase_idx
    
    result2.append([
        session, brand,
        has_view,
        valid_view_cart,
        valid_cart_purchase
    ])

funnel_df2 = pd.DataFrame(result2, columns=[
    'user_session', 'brand',
    'view', 'view_to_cart', 'cart_to_purchase'
])

## 브랜드 단위 세션 퍼널 - brand_funnel

- user_session + brand : 브랜드 단위 세션 퍼널
    - 브랜드 관심도/브랜드 내 전환

In [ ]:
### 브랜드별 세션 단위의 행동 집계

In [ ]:
# 세션별 행동 집계 (10월)
brand_session_activity = (
    df_final.groupby(['user_session', 'brand'])
    .agg(
        view_count=('event_type', lambda x: (x == 'view').sum()),
        cart_count=('event_type', lambda x: (x == 'cart').sum()),
        purchase_count=('event_type', lambda x: (x == 'purchase').sum())
    )
    .reset_index()
)

In [ ]:
# 세션별 행동 집계 (11월)
brand_session_activity2 = (
    df2_final.groupby(['user_session', 'brand'])
    .agg(
        view_count=('event_type', lambda x: (x == 'view').sum()),
        cart_count=('event_type', lambda x: (x == 'cart').sum()),
        purchase_count=('event_type', lambda x: (x == 'purchase').sum())
    )
    .reset_index()
)

### 브랜드별 퍼널 집계 및 전환율 계산
- 브랜드별 퍼널 집계 및 전환율 계산

In [ ]:
### 10월
# 브랜드별 세션 기반 전환 집계
brand_funnel = (
    funnel_df.groupby('brand')
    .agg(
        view_sessions=('view', 'sum'),
        view_to_cart_sessions=('view_to_cart', 'sum'),
        cart_to_purchase_sessions=('cart_to_purchase', 'sum')
    )
    .reset_index()
)

# 전환율 계산
brand_funnel['view_to_cart_rate'] = np.where(
    brand_funnel['view_sessions'] > 0,
    brand_funnel['view_to_cart_sessions'] / brand_funnel['view_sessions'],
    0
)

brand_funnel['cart_to_purchase_rate'] = np.where(
    brand_funnel['view_to_cart_sessions'] > 0,
    brand_funnel['cart_to_purchase_sessions'] / brand_funnel['view_to_cart_sessions'],
    0
)

brand_funnel['total_conversion_rate'] = np.where(
    brand_funnel['view_sessions'] > 0,
    brand_funnel['cart_to_purchase_sessions'] / brand_funnel['view_sessions'],
    0
)

# 이탈률
brand_funnel['drop_view_to_cart'] = 1 - brand_funnel['view_to_cart_rate']
brand_funnel['drop_cart_to_purchase'] = 1 - brand_funnel['cart_to_purchase_rate']

# 표본 적은 브랜드 제거
brand_funnel = brand_funnel[brand_funnel['view_sessions'] >= 100].copy()

# 매출 데이터 결합
brand_revenue = (
    df_final[df_final['event_type'] == 'purchase']
    .groupby('brand')
    .agg(
        revenue=('price', 'sum'),
        purchase_count=('price', 'count'),
        avg_price=('price', 'mean')
    )
    .reset_index()
)

brand_analysis = brand_funnel.merge(
    brand_revenue,
    on='brand',
    how='left'
).fillna(0)

# 보기 좋게 정렬
brand_analysis = brand_analysis.sort_values(
    by='total_conversion_rate',
    ascending=False
)

# 퍼센트 변환
rate_cols = [
    'view_to_cart_rate',
    'cart_to_purchase_rate',
    'total_conversion_rate',
    'drop_view_to_cart',
    'drop_cart_to_purchase'
]

brand_analysis[rate_cols] = brand_analysis[rate_cols] * 100

brand_analysis.head(20)

In [ ]:
### 11월
# 브랜드별 세션 기반 전환 집계
brand_funnel2 = (
    funnel_df2.groupby('brand')
    .agg(
        view_sessions=('view', 'sum'),
        view_to_cart_sessions=('view_to_cart', 'sum'),
        cart_to_purchase_sessions=('cart_to_purchase', 'sum')
    )
    .reset_index()
)

# 전환율 계산
brand_funnel2['view_to_cart_rate'] = np.where(
    brand_funnel2['view_sessions'] > 0,
    brand_funnel2['view_to_cart_sessions'] / brand_funnel2['view_sessions'],
    0
)

brand_funnel2['cart_to_purchase_rate'] = np.where(
    brand_funnel2['view_to_cart_sessions'] > 0,
    brand_funnel2['cart_to_purchase_sessions'] / brand_funnel2['view_to_cart_sessions'],
    0
)

brand_funnel2['total_conversion_rate'] = np.where(
    brand_funnel2['view_sessions'] > 0,
    brand_funnel2['cart_to_purchase_sessions'] / brand_funnel2['view_sessions'],
    0
)

# 이탈률
brand_funnel2['drop_view_to_cart'] = 1 - brand_funnel2['view_to_cart_rate']
brand_funnel2['drop_cart_to_purchase'] = 1 - brand_funnel2['cart_to_purchase_rate']

# 표본 적은 브랜드 제거
brand_funnel2 = brand_funnel2[brand_funnel2['view_sessions'] >= 100].copy()

# 매출 데이터 결합
brand_revenue2 = (
    df2_final[df2_final['event_type'] == 'purchase']
    .groupby('brand')
    .agg(
        revenue=('price', 'sum'),
        purchase_count=('price', 'count'),
        avg_price=('price', 'mean')
    )
    .reset_index()
)

brand_analysis2 = brand_funnel2.merge(
    brand_revenue2,
    on='brand',
    how='left'
).fillna(0)

# 보기 좋게 정렬
brand_analysis2 = brand_analysis2.sort_values(
    by='total_conversion_rate',
    ascending=False
)

# 퍼센트 변환
rate_cols = [
    'view_to_cart_rate',
    'cart_to_purchase_rate',
    'total_conversion_rate',
    'drop_view_to_cart',
    'drop_cart_to_purchase'
]

brand_analysis2[rate_cols] = brand_analysis2[rate_cols] * 100

brand_analysis2.head(20)

In [ ]:
### 10월
# 브랜드별 퍼널 집계 및 전환율 계산
brand_funnel = (
    funnel_df.groupby('brand')
    .agg(
        view_sessions=('view', 'sum'),
        view_to_cart_sessions=('view_to_cart', 'sum'),
        cart_to_purchase_sessions=('cart_to_purchase', 'sum')
    )
    .reset_index()
)

# 전환율 계산
brand_funnel['view_to_cart_rate'] = np.where(
    brand_funnel['view_sessions'] > 0,
    brand_funnel['view_to_cart_sessions'] / brand_funnel['view_sessions'],
    0
)

brand_funnel['cart_to_purchase_rate'] = np.where(
    brand_funnel['view_to_cart_sessions'] > 0,
    brand_funnel['cart_to_purchase_sessions'] / brand_funnel['view_to_cart_sessions'],
    0
)

brand_funnel['total_conversion_rate'] = np.where(
    brand_funnel['view_sessions'] > 0,
    brand_funnel['cart_to_purchase_sessions'] / brand_funnel['view_sessions'],
    0
)

# 이탈률
brand_funnel['drop_view_to_cart'] = 1 - brand_funnel['view_to_cart_rate']
brand_funnel['drop_cart_to_purchase'] = 1 - brand_funnel['cart_to_purchase_rate']

# 표본 적은 브랜드 제거
brand_funnel = brand_funnel[brand_funnel['view_sessions'] >= 100].copy()

# 매출 데이터 결합
brand_revenue = (
    df_final[df_final['event_type'] == 'purchase']
    .groupby('brand')
    .agg(
        revenue=('price', 'sum'),
        purchase_count=('price', 'count'),
        avg_price=('price', 'mean')
    )
    .reset_index()
)

brand_analysis = brand_funnel.merge(
    brand_revenue,
    on='brand',
    how='left'
).fillna(0)

# 보기 좋게 정렬
brand_analysis = brand_analysis.sort_values(
    by='total_conversion_rate',
    ascending=False
)

# 퍼센트 변환
rate_cols = [
    'view_to_cart_rate',
    'cart_to_purchase_rate',
    'total_conversion_rate',
    'drop_view_to_cart',
    'drop_cart_to_purchase'
]

brand_analysis[rate_cols] = brand_analysis[rate_cols] * 100

brand_analysis.head(20)

In [ ]:
### 11월
# 브랜드별 퍼널 집계 및 전환율 계산
brand_funnel2 = (
    funnel_df2.groupby('brand')
    .agg(
        view_sessions=('view', 'sum'),
        view_to_cart_sessions=('view_to_cart', 'sum'),
        cart_to_purchase_sessions=('cart_to_purchase', 'sum')
    )
    .reset_index()
)

# 전환율 계산
brand_funnel2['view_to_cart_rate'] = np.where(
    brand_funnel2['view_sessions'] > 0,
    brand_funnel2['view_to_cart_sessions'] / brand_funnel2['view_sessions'],
    0
)

brand_funnel2['cart_to_purchase_rate'] = np.where(
    brand_funnel2['view_to_cart_sessions'] > 0,
    brand_funnel2['cart_to_purchase_sessions'] / brand_funnel2['view_to_cart_sessions'],
    0
)

brand_funnel2['total_conversion_rate'] = np.where(
    brand_funnel2['view_sessions'] > 0,
    brand_funnel2['cart_to_purchase_sessions'] / brand_funnel2['view_sessions'],
    0
)

# 이탈률
brand_funnel2['drop_view_to_cart'] = 1 - brand_funnel2['view_to_cart_rate']
brand_funnel2['drop_cart_to_purchase'] = 1 - brand_funnel2['cart_to_purchase_rate']

# 표본 적은 브랜드 제거
brand_funnel2 = brand_funnel2[brand_funnel2['view_sessions'] >= 100].copy()

# 매출 데이터 결합
brand_revenue2 = (
    df2_final[df2_final['event_type'] == 'purchase']
    .groupby('brand')
    .agg(
        revenue=('price', 'sum'),
        purchase_count=('price', 'count'),
        avg_price=('price', 'mean')
    )
    .reset_index()
)

brand_analysis2 = brand_funnel2.merge(
    brand_revenue2,
    on='brand',
    how='left'
).fillna(0)

# 보기 좋게 정렬
brand_analysis2 = brand_analysis2.sort_values(
    by='total_conversion_rate',
    ascending=False
)

# 퍼센트 변환
rate_cols = [
    'view_to_cart_rate',
    'cart_to_purchase_rate',
    'total_conversion_rate',
    'drop_view_to_cart',
    'drop_cart_to_purchase'
]

brand_analysis2[rate_cols] = brand_analysis2[rate_cols] * 100

brand_analysis2.head(20)

# 기타

In [ ]:
# 전처리 근거 및 EDA는 최종 코드 파일에만 넣고 전처리 파일에는 넣지 않았습니다!
# 풀 이후 copy해서 파일명 변경 후 작업 붙여넣어주시면 됩니다! 화이팅입니다~!

In [ ]:
df_final.to_csv('eCommerce_smartphone_10월.csv', index=False, encoding='utf-8-sig')
df2_final.to_csv('eCommerce_smartphone_11월.csv', index=False, encoding='utf-8-sig')

In [ ]:
funnel_df.to_csv('eCommerce_smartphone_10월_funnel.csv', index=False, encoding='utf-8-sig')
funnel_df2.to_csv('eCommerce_smartphone_11월_funnel.csv', index=False, encoding='utf-8-sig')

In [ ]:
brand_funnel.to_csv('eCommerce_smartphone_10월_brand.csv', index=False, encoding='utf-8-sig')
brand_funnel2.to_csv('eCommerce_smartphone_11월_brand.csv', index=False, encoding='utf-8-sig')

# 정보
### [데이터 흐름도]
- df - df_final - funnel_df (퍼널 분석 시) / brand_funnel (브랜드 단위 분석 시)

### [데이터 딕셔너리]
| 파생변수명                   | 의미                          |
| ----------------------- | --------------------------- |
| `hour`                  | 이벤트 발생 시간대(0~23시)           |
| `day_of_week`           | 이벤트 발생 요일 정보                |
| `time_segment`          | 시간대를 새벽/오전/오후/저녁으로 구간화한 변수  |
| `is_purchase`           | 구매 이벤트 여부(1=구매, 0=구매 외)       |
| `is_cart`               | 장바구니 이벤트 여부(1=장바구니, 0=장바구니 외)   |
| `gap_sec`               | 동일 상품 재방문·재구매까지 걸린 시간 간격(초) |
| `buy_cnt`               | 세션 내 동일 상품 구매 횟수            |
| `view_to_cart_rate`     | 조회 대비 장바구니 전환율              |
| `cart_to_purchase_rate` | 장바구니 대비 구매 전환율              |
| `total_conversion_rate` | 조회 대비 최종 구매 전환율             |
| `drop_view_to_cart`     | 조회 후 장바구니 진입 전 이탈률          |
| `drop_cart_to_purchase` | 장바구니 후 구매 전 이탈률             |
| `revenue`               | 총 구매 금액(매출)                 |
| `purchase_count`        | 총 구매 건수                     |
| `avg_price`             | 평균 구매 가격                    |
| `is_outlier`            | 비정상적 반복 구매 여부(이상치 플래그)      |
| `is_price_error`        | 가격 오류·결측·극단값 여부             |


# 앞으로 진행할 통계 분석
- 단계별 전환율 분석
    - 전환율(단계별 소비자 행동의 전환 비중)
        - 메인 퍼널 전환에 영향을 미치는 주요 독립 변수 간의 상관성 분석
        - 유저 분포 / 리텐션 커브 / 요일별 흐름 / 구매~이탈 소요 시간 코호트 분석
    - 단계별 병목 구간 특정 및 이탈 주요 영향 변수 탐색
    - 시계열 패턴 및 구매 도달시간으로 교차 탐색 빈도 추적
        - 시간대별/요일별/월별 소비자의 교차 탐색 빈도 추적
        - 구매 고려 시간(12h/24h/36h) & 최종 구매 도달시간
- 고객 세그먼트 분석
    - 세션 내 행동 데이터를 바탕으로 소비자 유형 그룹(탐색형 vs 충성형 vs 대중형 고객 심리 세그먼트)과 앞서 만든 시간대별 파생변수들과 결합하여 특징과 행동 패턴 파악 (전환 속도, 주요 구매 시간대 등)
- 브랜드 세그먼트 분석
    - 비즈니스 관점에서의 브랜드 세그먼트(시장 지배형 vs 프리미엄 수익형 vs 비교형)
    - 소비자의 ‘브랜드 교차 탐색 활동’ 추적 및 시간대별 교차탐색빈도 분석 결과와 비교